# Laboratorio Minería de Datos: Regresión Lineal Múltiple (CRISP-DM)
Tres ejercicios: **Precio del dólar**, **Nivel de glucosa** y **Consumo de energía**.

Fases CRISP-DM que cubre este cuaderno:
1. Comprensión de los datos
2. Preparación de los datos (limpieza)
3. Modelado (regresión lineal múltiple)
4. Evaluación (MSE, RMSE, R²)
5. Despliegue (exportar `.pkl` + app web)

## Paso 1. Librerías y configuración

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
os.makedirs("modelos", exist_ok=True)
os.makedirs("graficas", exist_ok=True)

# Definición de los tres ejercicios
EJERCICIOS = {
    "dolar":   {"archivo": "dolar_data.csv",   "features": ["Dia", "Inflacion", "Tasa_interes"],
                "target": "Precio_Dolar",    "unidad": "COP",   "titulo": "Dólar"},
    "glucosa": {"archivo": "glucosa_data.csv", "features": ["Edad", "IMC", "Actividad_Fisica"],
                "target": "Nivel_Glucosa",   "unidad": "mg/dL", "titulo": "Glucosa"},
    "energia": {"archivo": "energia_data.csv", "features": ["Temperatura", "Hora", "Dia_Semana"],
                "target": "Consumo_Energia", "unidad": "kWh",   "titulo": "Energía"},
}
CONFIG = {}      # coeficientes para la app web
RESUMEN = []     # tabla final de métricas

## Paso 2. Subir los 3 archivos CSV
Selecciona `dolar_data.csv`, `glucosa_data.csv` y `energia_data.csv`.

In [ ]:
try:
    from google.colab import files
    files.upload()
except ImportError:
    print("No estás en Colab: se usarán los CSV de la carpeta actual.")

## Paso 3. Comprensión de los datos

In [ ]:
DATOS = {}
for k, e in EJERCICIOS.items():
    df = pd.read_csv(e["archivo"])
    DATOS[k] = df
    print("=" * 60)
    print(f"{e['titulo'].upper()}  |  filas: {df.shape[0]}  columnas: {df.shape[1]}")
    print("=" * 60)
    print(df.head(), "\n")
    print(df.dtypes, "\n")
    print(df.describe().round(3), "\n")

## Paso 4. Preparación de los datos (limpieza)
Para cada dataset se revisa:
1. **Tipos de dato**: que todas las columnas sean numéricas.
2. **Valores nulos**: se eliminan las filas incompletas.
3. **Duplicados**: se eliminan.
4. **Outliers (regla IQR)**: se **detectan y visualizan**, pero **no se eliminan** (son pocos y plausibles).

In [ ]:
def limpieza(df, nombre):
    print("=" * 60)
    print(f"LIMPIEZA: {nombre.upper()}")
    print("=" * 60)
    n0 = len(df)

    # 1) Tipos de dato: forzar a numérico (lo no convertible pasa a NaN)
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # 2) Nulos
    nulos = df.isnull().sum()
    print("Nulos por columna:\n", nulos.to_string())
    df = df.dropna()

    # 3) Duplicados
    dup = df.duplicated().sum()
    print("Filas duplicadas:", dup)
    df = df.drop_duplicates()

    # 4) Outliers por IQR (solo se reportan)
    print("\nOutliers (criterio IQR, no se eliminan):")
    for c in df.columns:
        q1, q3 = df[c].quantile([0.25, 0.75])
        iqr = q3 - q1
        n = ((df[c] < q1 - 1.5 * iqr) | (df[c] > q3 + 1.5 * iqr)).sum()
        print(f"  {c:18s}: {n} ({n / len(df) * 100:.1f}%)")

    fig, axs = plt.subplots(1, len(df.columns), figsize=(3.2 * len(df.columns), 3.2))
    for ax, c in zip(axs, df.columns):
        sns.boxplot(y=df[c], ax=ax, color="#93c5fd")
        ax.set_title(c)
    plt.suptitle(f"Boxplots - {nombre}", y=1.03)
    plt.tight_layout()
    plt.savefig(f"graficas/{nombre}_boxplots.png", dpi=120, bbox_inches="tight")
    plt.show()

    print(f"\nFilas antes: {n0} | filas después: {len(df)}\n")
    return df.reset_index(drop=True)

LIMPIOS = {k: limpieza(DATOS[k].copy(), k) for k in EJERCICIOS}

## Paso 5. Visualización de relaciones
Mapa de correlación y gráficas de dispersión de cada variable independiente contra la dependiente.

In [ ]:
def graficar(df, nombre, features, target):
    # Correlación
    plt.figure(figsize=(5.5, 4))
    sns.heatmap(df[features + [target]].corr(), annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
    plt.title(f"Correlación - {nombre}")
    plt.tight_layout()
    plt.savefig(f"graficas/{nombre}_correlacion.png", dpi=120)
    plt.show()

    # Dispersión: cada variable independiente vs dependiente
    fig, axs = plt.subplots(1, len(features), figsize=(5 * len(features), 4))
    for ax, f in zip(axs, features):
        sns.regplot(x=df[f], y=df[target], ax=ax, scatter_kws={"alpha": 0.35, "s": 12},
                    line_kws={"color": "red"})
        ax.set_title(f"{f} vs {target}")
    plt.tight_layout()
    plt.savefig(f"graficas/{nombre}_dispersion.png", dpi=120)
    plt.show()

for k, e in EJERCICIOS.items():
    print("#" * 60, "\n", e["titulo"].upper(), "\n" + "#" * 60)
    graficar(LIMPIOS[k], k, e["features"], e["target"])

## Paso 6. Modelado y evaluación
Se divide 80 % entrenamiento / 20 % prueba, se ajusta `LinearRegression` y se evalúa con **MSE, RMSE y R²**.
Para comparar la importancia de cada variable se usa el **coeficiente estandarizado**
(`coef × desv.est(X) / desv.est(y)`), porque los coeficientes crudos dependen de las unidades.

In [ ]:
def entrenar(nombre, e):
    df = LIMPIOS[nombre]
    features, target = e["features"], e["target"]
    X, y = df[features], df[target]

    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    modelo = LinearRegression().fit(X_tr, y_tr)

    # Evaluación
    pred = modelo.predict(X_te)
    mse = mean_squared_error(y_te, pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_te, pred)

    # Coeficientes
    coef = pd.DataFrame({
        "Variable": features,
        "Coeficiente": modelo.coef_,
        "Coef_estandarizado": modelo.coef_ * X.std().values / y.std(),
    })
    coef["Impacto_abs"] = coef["Coef_estandarizado"].abs()
    coef = coef.sort_values("Impacto_abs", ascending=False).reset_index(drop=True)
    top = coef.loc[0, "Variable"]

    print("=" * 60)
    print(f"MODELO: {e['titulo'].upper()}")
    print("=" * 60)
    ecuacion = f"{target} = {modelo.intercept_:.4f}" + "".join(
        f" {c:+.4f}*{f}" for c, f in zip(modelo.coef_, features))
    print("Ecuación:", ecuacion)
    print(f"MSE : {mse:.4f}\nRMSE: {rmse:.4f}\nR²  : {r2:.4f}\n")
    print(coef.round(4).to_string(index=False))
    print(f"\n➡ Variable de mayor impacto: {top}\n")

    # Real vs predicho
    plt.figure(figsize=(5, 4.5))
    plt.scatter(y_te, pred, alpha=0.4, s=12)
    lim = [min(y_te.min(), pred.min()), max(y_te.max(), pred.max())]
    plt.plot(lim, lim, "r--")
    plt.xlabel("Real"); plt.ylabel("Predicho")
    plt.title(f"Real vs Predicho - {nombre} (R²={r2:.3f})")
    plt.tight_layout()
    plt.savefig(f"graficas/{nombre}_real_vs_predicho.png", dpi=120)
    plt.show()

    # Residuos
    plt.figure(figsize=(5, 3.5))
    plt.scatter(pred, y_te - pred, alpha=0.4, s=12)
    plt.axhline(0, color="r", ls="--")
    plt.xlabel("Predicho"); plt.ylabel("Residuo"); plt.title(f"Residuos - {nombre}")
    plt.tight_layout()
    plt.savefig(f"graficas/{nombre}_residuos.png", dpi=120)
    plt.show()

    RESUMEN.append({"Ejercicio": e["titulo"], "MSE": mse, "RMSE": rmse, "R2": r2,
                    "Variable_mayor_impacto": top})

    # Configuración para la app web
    CONFIG[nombre] = {
        "titulo": e["titulo"], "features": features, "target": target, "unidad": e["unidad"],
        "intercept": float(modelo.intercept_), "coef": [float(c) for c in modelo.coef_],
        "rango": {f: [float(df[f].min()), float(df[f].max())] for f in features},
        "r2": float(r2), "rmse": float(rmse),
    }
    return modelo

MODELOS = {k: entrenar(k, e) for k, e in EJERCICIOS.items()}

## Paso 7. (Opcional) Significancia estadística de los coeficientes
Los valores-p indican si cada variable aporta de forma estadísticamente significativa (p < 0.05).

In [ ]:
try:
    import statsmodels.api as sm
    for k, e in EJERCICIOS.items():
        df = LIMPIOS[k]
        res = sm.OLS(df[e["target"]], sm.add_constant(df[e["features"]])).fit()
        print("=" * 60, "\n", e["titulo"].upper(), "\n" + "=" * 60)
        print(res.summary().tables[1], "\n")
except ImportError:
    print("statsmodels no está instalado (en Colab sí viene instalado).")

## Paso 8. Tabla resumen de los tres modelos

In [ ]:
resumen = pd.DataFrame(RESUMEN).round(4)
display(resumen) if "display" in globals() else print(resumen)
resumen.to_csv("modelos/resumen_metricas.csv", index=False)

## Paso 9. Exportar los modelos (`.pkl`) y verificar que se cargan
Se guardan con **joblib**. Además se exporta `config.json` (intercepto y coeficientes) que usará la app web en Vercel,
porque scikit-learn es demasiado pesado para el entorno serverless.

In [ ]:
for k, m in MODELOS.items():
    joblib.dump(m, f"modelos/modelo_{k}.pkl")

with open("modelos/config.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2, ensure_ascii=False)

# Verificación: cargar el .pkl y comparar con la fórmula manual
ejemplos = {"dolar": [[300, 0.02, 5.0]], "glucosa": [[50, 25, 4]], "energia": [[25, 12, 3]]}
for k, x in ejemplos.items():
    m = joblib.load(f"modelos/modelo_{k}.pkl")
    c = CONFIG[k]
    manual = c["intercept"] + sum(a * b for a, b in zip(c["coef"], x[0]))
    print(f"{k:8s} sklearn = {m.predict(pd.DataFrame(x, columns=c['features']))[0]:.4f} | fórmula = {manual:.4f}")

## Paso 10. Descargar todo (modelos + gráficas)

In [ ]:
!zip -r resultados.zip modelos graficas
try:
    from google.colab import files
    files.download("resultados.zip")
except ImportError:
    pass